In [1]:
import polars as pl
import numpy as np
import json
from pathlib import Path

pl.enable_string_cache()

In [2]:
INPUT_PATH = Path("../data/processed/nyc_311_cleaned.csv")
OUTPUT_PATH = Path("../data/training/nyc_311_features.csv")
MAPPING_DIR = Path("../data/artifacts/mappings")

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
MAPPING_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
df = pl.read_csv(INPUT_PATH, try_parse_dates=True)
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns}")

Shape: (2904438, 14)
Columns: ['unique_key', 'created_date', 'closed_date', 'agency', 'agency_name', 'complaint_type', 'descriptor', 'location_type', 'borough', 'latitude', 'longitude', 'resolution_minutes', 'hour', 'weekday']


In [4]:
df = df.with_columns(
    pl.when(pl.col("resolution_minutes") < 30).then(0)
    .when(pl.col("resolution_minutes") < 60).then(1)
    .when(pl.col("resolution_minutes") < 150).then(2)
    .when(pl.col("resolution_minutes") < 420).then(3)
    .when(pl.col("resolution_minutes") < 1440).then(4)
    .when(pl.col("resolution_minutes") < 4320).then(5)
    .when(pl.col("resolution_minutes") < 10080).then(6)
    .otherwise(7)
    .alias("target_bucket")
)

In [5]:
df = df.with_columns([
    pl.col("created_date").dt.hour().alias("hour"),
    pl.col("created_date").dt.minute().alias("minute"),
    pl.col("created_date").dt.second().alias("second"),
])

df = df.with_columns(
    ((pl.col("hour") * 3600) + (pl.col("minute") * 60) + pl.col("second")).alias("seconds_since_midnight")
)

df = df.with_columns(
    (pl.col("seconds_since_midnight") / 86400).alias("day_fraction")
)

df = df.with_columns([
    (2 * np.pi * pl.col("day_fraction")).sin().round(4).alias("time_sin"),
    (2 * np.pi * pl.col("day_fraction")).cos().round(4).alias("time_cos"),
])

In [6]:
df = df.with_columns(
    (pl.col("created_date").dt.weekday() - 1).alias("weekday")
)

df = df.with_columns(
    (pl.col("weekday") * 86400 + pl.col("seconds_since_midnight")).alias("week_seconds")
)

df = df.with_columns(
    (pl.col("week_seconds") / (7 * 86400)).alias("week_fraction")
)

df = df.with_columns([
    (2 * np.pi * pl.col("week_fraction")).sin().round(4).alias("week_sin"),
    (2 * np.pi * pl.col("week_fraction")).cos().round(4).alias("week_cos"),
])

df = df.with_columns(
    pl.when(pl.col("weekday").is_in([5, 6])).then(1).otherwise(0).alias("is_weekend")
)

In [7]:
latitude_mean: float = df["latitude"].mean() # type: ignore
latitude_std: float = df["latitude"].std() # type: ignore
longitude_mean: float = df["longitude"].mean() # type: ignore
longitude_std: float = df["longitude"].std() # type: ignore

df = df.with_columns([
    ((pl.col("latitude") - latitude_mean) / latitude_std).round(6).alias("latitude_z"),
    ((pl.col("longitude") - longitude_mean) / longitude_std).round(6).alias("longitude_z"),
])

scaler_metadata = {
    "latitude_mean": latitude_mean,
    "latitude_std": latitude_std,
    "longitude_mean": longitude_mean,
    "longitude_std": longitude_std,
}

with open(MAPPING_DIR / "scaler_metadata.json", "w") as f:
    json.dump(scaler_metadata, f, indent=2)

print("Scaler metadata saved.")
print(scaler_metadata)

Scaler metadata saved.
{'latitude_mean': 40.738379091362894, 'latitude_std': 0.08834923279601656, 'longitude_mean': -73.92112745125998, 'longitude_std': 0.07669633444440958}


In [8]:
df = df.with_columns(
    (pl.col("complaint_type") + "|" + pl.col("descriptor") + "|" + pl.col("location_type")).alias("combo")
)

print(f"Unique combos retained: {df['combo'].n_unique():,}")

Unique combos retained: 522


In [9]:
agency_vals = df["agency"].unique().sort().to_list()
agency_mapping = {val: i for i, val in enumerate(agency_vals)}

name_map = {}
for val, idx in agency_mapping.items():
    full_name = df.filter(pl.col("agency") == val)["agency_name"].unique().to_list()
    name_map[idx] = full_name[0] if full_name else val

df = df.with_columns(
    pl.col("agency")
    .replace(agency_mapping)
    .cast(pl.UInt16)
    .alias("agency_id")
).drop("agency")

with open(MAPPING_DIR / "agency_map.json", "w") as f:
    json.dump(name_map, f, indent=2)

print(f"Agency: {len(agency_vals)} categories → agency_id")

Agency: 14 categories → agency_id


In [10]:
complaint_vals = df["complaint_type"].unique().sort().to_list()
complaint_mapping = {val: i for i, val in enumerate(complaint_vals)}

df = df.with_columns(
    pl.col("complaint_type")
    .replace(complaint_mapping)
    .cast(pl.UInt16)
    .alias("complaint_id")
).drop("complaint_type")

with open(MAPPING_DIR / "complaint_map.json", "w") as f:
    json.dump({v: k for k, v in complaint_mapping.items()}, f, indent=2)

print(f"Complaint Type: {len(complaint_vals)} categories → complaint_id")

Complaint Type: 110 categories → complaint_id


In [11]:
descriptor_vals = df["descriptor"].unique().sort().to_list()
descriptor_mapping = {val: i for i, val in enumerate(descriptor_vals)}

df = df.with_columns(
    pl.col("descriptor")
    .replace(descriptor_mapping)
    .cast(pl.UInt16)
    .alias("descriptor_id")
).drop("descriptor")

with open(MAPPING_DIR / "descriptor_map.json", "w") as f:
    json.dump({v: k for k, v in descriptor_mapping.items()}, f, indent=2)

print(f"Descriptor: {len(descriptor_vals)} categories → descriptor_id")

Descriptor: 397 categories → descriptor_id


In [12]:
location_vals = df["location_type"].unique().sort().to_list()
location_mapping = {val: i for i, val in enumerate(location_vals)}

df = df.with_columns(
    pl.col("location_type")
    .replace(location_mapping)
    .cast(pl.UInt16)
    .alias("location_id")
).drop("location_type")

with open(MAPPING_DIR / "location_map.json", "w") as f:
    json.dump({v: k for k, v in location_mapping.items()}, f, indent=2)

print(f"Location Type: {len(location_vals)} categories → location_id")

Location Type: 41 categories → location_id


In [13]:
borough_vals = df["borough"].unique().sort().to_list()
borough_mapping = {val: i for i, val in enumerate(borough_vals)}

df = df.with_columns(
    pl.col("borough")
    .replace(borough_mapping)
    .cast(pl.UInt16)
    .alias("borough_id")
).drop("borough")

with open(MAPPING_DIR / "borough_map.json", "w") as f:
    json.dump({v: k for k, v in borough_mapping.items()}, f, indent=2)

print(f"Borough: {len(borough_vals)} categories → borough_id")

Borough: 6 categories → borough_id


In [14]:
combo_vals = df["combo"].unique().sort().to_list()
combo_mapping = {val: i for i, val in enumerate(combo_vals)}

df = df.with_columns(
    pl.col("combo")
    .replace(combo_mapping)
    .cast(pl.UInt16)
    .alias("combo_id")
).drop("combo")

with open(MAPPING_DIR / "combo_map.json", "w") as f:
    json.dump({v: k for k, v in combo_mapping.items()}, f, indent=2)

print(f"Combo: {len(combo_vals)} categories → combo_id")

Combo: 522 categories → combo_id


In [15]:
columns_to_drop = [
    "unique_key",
    "created_date",
    "closed_date",
    "agency_name",
    "resolution_minutes",
    "hour",
    "minute",
    "second",
    "seconds_since_midnight",
    "day_fraction",
    "week_seconds",
    "week_fraction",
    "weekday",
    "borough",
    "latitude",
    "longitude",
]

df = df.drop([c for c in columns_to_drop if c in df.columns])

print(f"Final shape: {df.shape}")
print(f"Final columns ({len(df.columns)}):")
print(df.columns)

Final shape: (2904438, 14)
Final columns (14):
['target_bucket', 'time_sin', 'time_cos', 'week_sin', 'week_cos', 'is_weekend', 'latitude_z', 'longitude_z', 'agency_id', 'complaint_id', 'descriptor_id', 'location_id', 'borough_id', 'combo_id']


In [16]:
df.write_csv(OUTPUT_PATH)
print(f"Exported {df.shape[0]:,} rows × {df.shape[1]} columns → {OUTPUT_PATH.resolve()}")

Exported 2,904,438 rows × 14 columns → ../data/training/nyc_311_features.csv


In [17]:
numeric_types = {
    pl.Int8, pl.Int16, pl.Int32, pl.Int64,
    pl.UInt8, pl.UInt16, pl.UInt32, pl.UInt64,
    pl.Float32, pl.Float64,
}

print("\n=== Verification ===")
print(f"All numeric: {all(t in numeric_types for t in df.dtypes)}")
print(f"Null count: {df.null_count().sum_horizontal().item()}")
print(f"\nTarget distribution:")
print(df["target_bucket"].value_counts().sort("target_bucket"))


=== Verification ===
All numeric: True
Null count: 0

Target distribution:
shape: (8, 2)
┌───────────────┬────────┐
│ target_bucket ┆ count  │
│ ---           ┆ ---    │
│ i32           ┆ u32    │
╞═══════════════╪════════╡
│ 0             ┆ 345868 │
│ 1             ┆ 312803 │
│ 2             ┆ 437404 │
│ 3             ┆ 361599 │
│ 4             ┆ 321267 │
│ 5             ┆ 433120 │
│ 6             ┆ 273722 │
│ 7             ┆ 418655 │
└───────────────┴────────┘


In [18]:
nonzero_per_row = (
    (df.drop("target_bucket") != 0)
    .sum_horizontal()
    .mean()
)
print(f"Average non-zero features per row (excluding target): {nonzero_per_row:.2f}")

Average non-zero features per row (excluding target): 12.03
